# P2 · Pipelines primer: Beam, xarray-beam and Spark

*A tool primer. It uses synthetic data, so it can be read before or alongside the project notebooks.*

A data pipeline is a program that moves data from where it arrives to where it is useful, transforming it on the way: decode, clean, join, aggregate, write. On one laptop, a loop over files does the job. At terabytes or petabytes, the same logic has to run on many machines at once, survive machines failing halfway, and often keep running forever as new data streams in. **Pipeline frameworks** exist so that you write the logic once and the framework handles the distribution, the retries and the bookkeeping.

This primer covers two frameworks with different mental models, plus a weather-specific extension of one of them.

- **Apache Beam** is a *programming model*: you describe a pipeline as a graph of transformations, and a separate **runner** executes it, on your laptop, on Google Cloud Dataflow, on Flink or on Spark. Its strength is one model for both **batch** (a fixed dataset) and **streaming** (data that keeps arriving), with careful handling of time.
- **xarray-beam** applies Beam to labelled arrays (xarray datasets split into chunks). Google built large weather datasets with it, including the cloud copy of ERA5 (ARCO-ERA5) and WeatherBench 2.
- **Apache Spark** is an *engine* for large tables. You write DataFrame or SQL operations, and Spark plans and runs them across a cluster. It is the default tool for tabular data at scale, and it also has a streaming mode.

The data is synthetic and small enough to run on a laptop in minutes, but every pattern here is the one used at scale.

> **In ML terms.** A pipeline framework is to data processing what a training framework is to model code: you declare the computation, and the framework handles parallelism, scheduling and failure. Beam is closest to a dataflow graph that can be compiled for different backends; Spark is closest to a distributed DataFrame library with a query optimizer.

In [1]:
# Setup: identical in every notebook. Data lives in ../data, which git ignores.
import os
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
os.makedirs(DATA_DIR, exist_ok=True)
print("Data directory:", DATA_DIR)

Data directory: /Users/ashu/Workspace/weather-data-lab/data


In [2]:
import glob, logging, shutil, subprocess, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)
from wxlab import synthetic

WORK = Path(DATA_DIR) / "primers" / "pipelines"
shutil.rmtree(WORK, ignore_errors=True); WORK.mkdir(parents=True)

cube = synthetic.make_cube(days=120)                    # 4 months, hourly, 80 x 120 grid
stations = synthetic.make_station_table(cube, n=100)    # 100 stations with planted faults
events = synthetic.make_event_stream(stations, hours=6, n_stations=5)
stations.to_parquet(WORK / "stations.parquet", index=False)
print(f"stations: {len(stations):,} rows; planted faults: {stations.attrs['faults']}")
print(f"events: {len(events)} readings, {events.late.sum()} of them arriving late")
stations.head(3)

stations: 282,240 rows; planted faults: [{'station_id': 'ST0007', 'fault': 'stuck', 'start': '2023-02-10 00:00:00', 'end': '2023-02-13 00:00:00'}, {'station_id': 'ST0042', 'fault': 'step +2.5', 'start': '2023-03-02 00:00:00', 'end': None}]
events: 180 readings, 16 of them arriving late


,station_id,time,air_temperature,latitude,longitude,elevation_m
0,ST0000,2023-01-01 00:00:00,15.921865,36.141859,-115.19201,0.0
1,ST0000,2023-01-01 01:00:00,14.980086,36.141859,-115.19201,0.0
2,ST0000,2023-01-01 02:00:00,14.227915,36.141859,-115.19201,0.0


Three datasets, each shaped like something real:

- **`stations`**: hourly temperature at 100 points, 4 months, as a table. Two faults are planted: station `ST0007` gets stuck on one value for three days, and station `ST0042` jumps 2.5 °C warmer halfway through and stays there. About 2% of rows are missing.
- **`events`**: six hours of readings from five stations, every 10 minutes, as a stream. Each reading has an **event time** (when it was measured) and an **arrival time** (when the pipeline received it). Most arrive within a minute or two; about 10% arrive 20–90 minutes late, as happens when a station's link drops and it resends its backlog.
- **`cube`**: the gridded field the stations were sampled from, for xarray-beam.

## Part 1 · Beam: pipelines as graphs

A Beam program builds a **Pipeline**: a graph whose edges are **PCollections** (distributed collections of elements, possibly unbounded) and whose nodes are **PTransforms** (operations that turn PCollections into PCollections). The `|` operator applies a transform. Nothing runs while you build the graph; the runner executes it when the `with` block ends.

The core transforms are few:

- **Map / FlatMap / Filter**: per-element functions (one-to-one, one-to-many, keep or drop).
- **GroupByKey**: collect all values with the same key. This is the step that moves data between machines (a **shuffle**).
- **CombinePerKey**: aggregate values per key (sum, mean, count). Beam can pre-aggregate on each machine before the shuffle, which is much cheaper than grouping everything first.

The pipeline below computes each station's daily mean temperature and writes it out. We select the classic `DirectRunner`, which runs everything in this process.

In [3]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

records = stations[stations.station_id.isin(["ST0000", "ST0001", "ST0042"])].to_dict("records")
out_prefix = str(WORK / "daily_means")

with beam.Pipeline(runner="DirectRunner") as p:
    (p
     | "Read" >> beam.Create(records)                                          # source
     | "Valid" >> beam.Filter(lambda r: np.isfinite(r["air_temperature"]))      # drop missing
     | "Key by station-day" >> beam.Map(lambda r: ((r["station_id"], str(r["time"])[:10]), r["air_temperature"]))
     | "Daily mean" >> beam.combiners.Mean.PerKey()                             # aggregate per key
     | "Format" >> beam.Map(lambda kv: f"{kv[0][0]},{kv[0][1]},{kv[1]:.2f}")
     | "Write" >> beam.io.WriteToText(out_prefix, file_name_suffix=".csv"))    # sink

files = sorted(glob.glob(out_prefix + "*"))
daily = pd.concat(pd.read_csv(f, names=["station_id", "day", "tmean"]) for f in files)
print(f"{len(files)} output file(s), {len(daily)} station-days")
daily.sort_values(["station_id", "day"]).head()

1 output file(s), 360 station-days


,station_id,day,tmean
0,ST0000,2023-01-01,11.21
1,ST0000,2023-01-02,10.32
2,ST0000,2023-01-03,8.14
3,ST0000,2023-01-04,9.90
4,ST0000,2023-01-05,12.69


Each step has a name (`"Daily mean"`), which matters at scale: the runner's monitoring UI shows every named step with its throughput and errors. The output is split into **shards** (files), because at scale many workers write in parallel; one file per worker is the norm.

Notice what the code does not contain: no threads, no file locking, no retry logic. The same pipeline, with `runner="DataflowRunner"` and a cloud bucket instead of local paths, runs on hundreds of machines.

## Part 2 · Beam streaming: event time, windows and watermarks

In batch, all data is present before the pipeline starts. In streaming it is not, which raises a question batch never asks: **when is a result final?** If we want each station's hourly average, when do we stop waiting for that hour's readings?

Beam's answer uses four ideas.

- **Event time vs processing time.** Results are grouped by when readings were *measured* (event time), not when they arrived. Otherwise a late reading would be counted in the wrong hour.
- **Windows** split the timeline into groups. **Fixed windows** are consecutive blocks (every hour); **sliding** windows overlap; **session** windows close after a gap in activity.
- The **watermark** is the runner's running estimate of "all readings with event time before T have probably arrived". When the watermark passes the end of a window, the window's result is emitted.
- **Allowed lateness and triggers** decide what happens to readings that arrive *after* the watermark has passed their window: drop them, or emit an updated result.

To make this concrete without a real stream, Beam's **TestStream** replays elements and watermark advances in a fixed order. Here, one station sends two readings in the first 10-minute window, the watermark moves past that window, then a third reading for the *first* window arrives late, and finally a reading for the second window arrives after the watermark has passed it as well.

In [4]:
from apache_beam.testing.test_stream import TestStream
from apache_beam.transforms import trigger, window
from apache_beam.transforms.window import TimestampedValue
from apache_beam.options.pipeline_options import StandardOptions

T0 = 1_700_000_400                      # a round timestamp: windows start at T0, T0 + 600, ...
TIMING = {0: "EARLY", 1: "ON_TIME", 2: "LATE", 3: "UNKNOWN"}

def replay(allowed_lateness_s):
    stream = (TestStream()
              .add_elements([TimestampedValue(("A", 10.0), T0 + 60), TimestampedValue(("A", 12.0), T0 + 300)])
              .advance_watermark_to(T0 + 600)                                  # window [0, 10 min) is now "complete"
              .add_elements([TimestampedValue(("A", 99.0), T0 + 120)])          # ...but a reading for it arrives late
              .advance_watermark_to(T0 + 1300)                                 # window [10, 20 min) is now complete too
              .add_elements([TimestampedValue(("A", 20.0), T0 + 700)])          # ...and its only reading arrives late
              .advance_watermark_to_infinity())
    opts = PipelineOptions(); opts.view_as(StandardOptions).streaming = True

    class Report(beam.DoFn):
        def process(self, kv, w=beam.DoFn.WindowParam, pane=beam.DoFn.PaneInfoParam):
            start = (int(w.start) - T0) // 60
            print(f"  window [{start:>2}, {start + 10:>2}) min: count={kv[1]}  pane={TIMING[pane.timing]}")

    late_trigger = trigger.AfterWatermark(late=trigger.AfterCount(1)) if allowed_lateness_s else trigger.AfterWatermark()
    with beam.Pipeline(runner="DirectRunner", options=opts) as p:
        (p | stream
           | beam.WindowInto(window.FixedWindows(600), trigger=late_trigger, allowed_lateness=allowed_lateness_s,
                             accumulation_mode=trigger.AccumulationMode.ACCUMULATING)
           | beam.combiners.Count.PerKey()
           | beam.ParDo(Report()))

print("No allowed lateness: late readings are dropped")
replay(0)
print("15 minutes of allowed lateness: late readings update the result")
replay(900)

No allowed lateness: late readings are dropped
  window [ 0, 10) min: count=2  pane=ON_TIME
15 minutes of allowed lateness: late readings update the result
  window [ 0, 10) min: count=2  pane=ON_TIME
  window [ 0, 10) min: count=3  pane=LATE
  window [10, 20) min: count=1  pane=LATE


With no allowed lateness, the first window is emitted once, **on time**, with 2 readings, and both late readings are silently dropped. The second window never appears at all, because its only reading arrived after the watermark had passed it.

With 15 minutes of allowed lateness and a trigger that fires on each late reading, the first window is emitted on time (2), then again as a **late pane** with the corrected count (3), and the second window appears as a late pane. Because the mode is *accumulating*, each late pane carries the full updated result rather than just the change.

This is the central trade-off of streaming: **latency against completeness**. Emit early and you may be wrong; wait longer and results are late; keep windows open for late data and you hold state (memory) for longer. For weather observations, which can arrive hours late after an outage, a common design is a fast streaming result plus a later batch job that recomputes each day once it is complete. Data that arrives even later is a **correction**, handled by reprocessing.

> **In ML terms.** Event time vs processing time is the difference between when a label was true and when you learned it. Training on features computed in processing time leaks information from the future; a watermark is the line at which you declare a training example final.

## Part 3 · xarray-beam: Beam for labelled arrays

Beam works on collections of elements. xarray-beam makes each element a **chunk of a labelled array**: a pair of a **key** (where the chunk sits, as offsets along each dimension) and a small xarray Dataset. Transforms then operate on chunks: split a dataset into chunks, **rechunk** them into a different layout, reduce along a dimension, and write the result into a Zarr store, each worker writing its own region.

Below, a 30-day piece of the cube is split into daily map-shaped chunks, rechunked into time-series tiles (the analysis layout from the arrays primer), and written to Zarr. A second pipeline computes the 30-day mean map.

In [5]:
import xarray_beam as xbeam

small = cube.isel(time=slice(0, 24 * 30)).drop_vars("elevation")
src_chunks = {"time": 24, "latitude": -1, "longitude": -1}         # -1 means the whole dimension
dst_chunks = {"time": -1, "latitude": 10, "longitude": 10}

# What one element looks like: a key (offsets) and a chunk.
key, chunk = next(iter(xbeam.split_chunks(xbeam.Key({"time": 0, "latitude": 0, "longitude": 0}),
                                          small, {"time": 24})))
print("key:", key, "| chunk sizes:", dict(chunk.sizes))

t0 = time.time()
with beam.Pipeline(runner="DirectRunner") as p:
    (p
     | xbeam.DatasetToChunks(small, src_chunks)
     | xbeam.Rechunk(dict(small.sizes), src_chunks, dst_chunks, itemsize=4)
     | xbeam.ChunksToZarr(str(WORK / "rechunked.zarr"), template=xbeam.make_template(small), zarr_chunks=dst_chunks))
print(f"rechunk pipeline: {time.time() - t0:.1f}s")

with beam.Pipeline(runner="DirectRunner") as p:
    (p
     | xbeam.DatasetToChunks(small, src_chunks)
     | xbeam.Mean("time")
     | xbeam.ChunksToZarr(str(WORK / "mean.zarr"), template=xbeam.make_template(small.mean("time")),
                          zarr_chunks={"latitude": -1, "longitude": -1}))

rechunked = xr.open_zarr(WORK / "rechunked.zarr").air_temperature
print("stored chunks:", rechunked.encoding["chunks"], "| identical to the source:",
      bool(np.array_equal(rechunked.values, small.air_temperature.values)))
diff = abs(xr.open_zarr(WORK / "mean.zarr").air_temperature - small.air_temperature.mean("time")).max()
print(f"largest difference from xarray's own mean: {float(diff):.1e} °C")

key: Key(offsets={'time': 0, 'latitude': 0, 'longitude': 0}) | chunk sizes: {'time': 24, 'latitude': 80, 'longitude': 120}


rechunk pipeline: 18.0s


stored chunks: (720, 10, 10) | identical to the source: True
largest difference from xarray's own mean: 1.9e-05 °C


On a laptop this is slower than doing the same thing in plain xarray, because every chunk passes through Beam's machinery. That overhead buys something at scale: the same pipeline runs on Dataflow across hundreds of workers, the rechunk is planned so no worker needs more than a bounded amount of memory, and writes go to independent regions of the Zarr store, so failed work can be retried safely. The tiny difference in the mean (about 10⁻⁵ °C) comes from summing in a different order, a reminder that floating-point results depend on how a computation is split.

## Part 4 · Spark: distributed tables

Spark's unit is the **DataFrame**: a table with a schema, split into **partitions** spread across a cluster. Like Beam, Spark is lazy: transformations (`select`, `filter`, `groupBy`) build a plan, and **actions** (`count`, `show`, `write`) run it. Before running, Spark's optimizer rewrites the plan: it pushes filters down to the file scan, reads only the columns it needs, and chooses join strategies.

Spark runs on the Java virtual machine, so it needs a Java runtime. The next cell finds one.

In [6]:
def find_java_home():
    """JAVA_HOME if set, else a JDK under ~/.jdks, else the macOS default."""
    if os.environ.get("JAVA_HOME"):
        return os.environ["JAVA_HOME"]
    for home in sorted(glob.glob(os.path.expanduser("~/.jdks/jdk-*/Contents/Home")), reverse=True):
        return home
    try:
        return subprocess.run(["/usr/libexec/java_home"], capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        raise RuntimeError("No Java found: install a JDK 17 or 21 and set JAVA_HOME")

os.environ["JAVA_HOME"] = find_java_home()
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder.master("local[4]").appName("primer")
         .config("spark.ui.enabled", "false")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.parquet(str(WORK / "stations.parquet"))
df.printSchema()
print(f"{df.count():,} rows in {df.rdd.getNumPartitions()} partition(s)")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/07 22:07:51 WARN Utils: Your hostname, Ashus-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 172.20.20.20 instead (on interface en0)
26/11/07 22:07:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/07 22:07:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- station_id: string (nullable = true)
 |-- time: timestamp_ntz (nullable = true)
 |-- air_temperature: float (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- elevation_m: float (nullable = true)



282,240 rows in 1 partition(s)


`local[4]` runs Spark on this machine with 4 worker threads; on a cluster the same code gets a cluster URL instead. `spark.sql.shuffle.partitions` sets how many pieces data is split into after a shuffle; the default of 200 suits a cluster, not a laptop.

### Aggregations and the shuffle

A `groupBy` needs all rows with the same key on the same machine, so Spark moves data between machines: the **shuffle**. It is usually the most expensive part of a job. `explain()` shows the plan; look for **Exchange**, which marks a shuffle, and notice the two **HashAggregate** steps: Spark computes partial averages on each partition *before* the shuffle and combines them after, the same idea as Beam's CombinePerKey.

In [7]:
daily = df.groupBy("station_id", F.to_date("time").alias("day")).agg(F.avg("air_temperature").alias("tmean"))
daily.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[station_id#0, _groupingexpression#26], functions=[avg(air_temperature#2)])
   +- Exchange hashpartitioning(station_id#0, _groupingexpression#26, 8), ENSURE_REQUIREMENTS, [plan_id=56]
      +- HashAggregate(keys=[station_id#0, _groupingexpression#26], functions=[partial_avg(air_temperature#2)])
         +- Project [station_id#0, air_temperature#2, cast(time#1 as date) AS _groupingexpression#26]
            +- FileScan parquet [station_id#0,time#1,air_temperature#2] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/ashu/Workspace/weather-data-lab/data/primers/pipelines/sta..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<station_id:string,time:timestamp_ntz,air_temperature:float>




### Window functions: finding a stuck sensor

**Window functions** (unrelated to Beam's streaming windows) compute a value for each row from neighbouring rows in the same group, for example the previous value of the same station. That is exactly what detecting a stuck sensor needs: mark each reading that equals the previous one, number the runs of repeated values, and keep runs longer than a threshold.

In [8]:
by_station = Window.partitionBy("station_id").orderBy("time")
stuck = (df
         .withColumn("repeat", (F.col("air_temperature") == F.lag("air_temperature").over(by_station)).cast("int"))
         .withColumn("run_id", F.sum(1 - F.col("repeat")).over(by_station))      # new id whenever the value changes
         .groupBy("station_id", "run_id")
         .agg(F.count("*").alias("hours"), F.min("time").alias("start"), F.first("air_temperature").alias("value"))
         .filter("hours >= 6")
         .orderBy(F.desc("hours")))
stuck.show()

+----------+------+-----+-------------------+---------+
|station_id|run_id|hours|              start|    value|
+----------+------+-----+-------------------+---------+
|    ST0007|   942|   71|2023-02-10 00:00:00|13.996944|
+----------+------+-----+-------------------+---------+



The check finds the planted fault: `ST0007` repeats one value for 71 hours (the 72-hour stuck period, minus readings lost to the random gaps). Nothing else repeats for six hours, because real temperatures always wiggle at least a little, but on a coarse sensor that reports whole degrees (as many airports do) the threshold would need care.

### Joins: comparing each station with the network

The planted step at `ST0042` is invisible in a single station's raw series, because the seasons change by far more than 2.5 °C. Comparing with other stations removes the shared weather. Here we compute each station's monthly mean, subtract the network's monthly mean (a **join** of two aggregates), and look at the change from one month to the next.

In [9]:
monthly = df.groupBy("station_id", F.date_format("time", "yyyy-MM").alias("month")).agg(F.avg("air_temperature").alias("tmean"))
network = monthly.groupBy("month").agg(F.avg("tmean").alias("network_mean"))
rel = (monthly.join(F.broadcast(network), "month")                       # small table: send a copy to every worker
              .withColumn("offset", F.col("tmean") - F.col("network_mean")))
by_month = Window.partitionBy("station_id").orderBy("month")
jumps = (rel.withColumn("change", F.col("offset") - F.lag("offset").over(by_month))
            .filter(F.abs("change") > 1.0).orderBy(F.desc(F.abs("change"))))
jumps.select("station_id", "month", F.round("offset", 2).alias("offset"), F.round("change", 2).alias("change")).show()

+----------+-------+------+------+
|station_id|  month|offset|change|
+----------+-------+------+------+
|    ST0042|2023-03|  3.17|  2.28|
+----------+-------+------+------+



Only `ST0042` changes its offset from the network by more than 1 °C between months, and the change appears in the month the step was planted. It is a little smaller than the planted 2.5 °C because the step starts on 2 March, so March's mean includes a day without it, and because of noise. This is a crude version of the comparison notebook 05 does properly against a weather model and neighbouring stations.

`F.broadcast` tells Spark the network table is small, so instead of shuffling both sides of the join, it copies the small table to every worker: a **broadcast join**. Choosing between broadcast and shuffle joins is one of the most common Spark performance decisions.

### Writing partitioned Parquet

Large tables are written **partitioned**: one folder per value of a column, such as month. A later query that filters on that column reads only the matching folders, which Spark calls **partition pruning**. The plan shows it as `PartitionFilters`.

In [10]:
out = WORK / "stations_by_month"
df.withColumn("month", F.date_format("time", "yyyy-MM")).write.mode("overwrite").partitionBy("month").parquet(str(out))
print(sorted(p.name for p in out.iterdir() if p.is_dir()))
one_month = spark.read.parquet(str(out)).filter("month = '2023-03'")
import re
plan = one_month._jdf.queryExecution().executedPlan().toString()
print("scan:", re.search(r"PartitionFilters: \[[^\]]*\]", plan).group(0))

['month=2023-01', 'month=2023-02', 'month=2023-03', 'month=2023-04']
scan: PartitionFilters: [isnotnull(month#166), (month#166 = 2023-03)]


### Structured Streaming

Spark's streaming mode treats a stream as a table that keeps growing and runs the same DataFrame code on each new piece (a **micro-batch**). Watermarks work like Beam's: `withWatermark("event_time", "30 minutes")` tells Spark that once it has seen a reading measured at 10:00, it may close windows that ended before 09:30, and drop readings for them that arrive afterwards.

We write the event stream as files in arrival order, one file per half hour of arrivals, and let Spark read them one at a time as if they were arriving live. The same aggregation (readings per station per hour) runs three times, with watermark delays of 5 minutes, 30 minutes and 2 hours.

In [11]:
events_dir = WORK / "events"
events_dir.mkdir()
for i, (_, batch) in enumerate(events.groupby(events.arrival_time.dt.floor("30min"))):
    batch.to_parquet(events_dir / f"batch_{i:03d}.parquet", index=False)
schema = spark.read.parquet(str(events_dir)).schema

rows = []
for delay in ["5 minutes", "30 minutes", "2 hours"]:
    name = "hourly_" + delay.split()[0]
    hourly = (spark.readStream.schema(schema).option("maxFilesPerTrigger", 1).parquet(str(events_dir))
              .withWatermark("event_time", delay)
              .groupBy(F.window("event_time", "1 hour"), "station_id")
              .agg(F.count("*").alias("readings")))
    query = (hourly.writeStream.outputMode("update").format("memory").queryName(name)
             .trigger(availableNow=True).start())
    query.awaitTermination()
    counted = spark.sql(f"select window, station_id, max(readings) as n from {name} group by 1, 2").toPandas()
    rows.append({"watermark delay": delay, "readings sent": len(events),
                 "counted": int(counted.n.sum()), "dropped as late": len(events) - int(counted.n.sum())})
print(f"{int(events.late.sum())} readings arrived 20-90 minutes after they were measured")
pd.DataFrame(rows)

16 readings arrived 20-90 minutes after they were measured


,watermark delay,readings sent,counted,dropped as late
0,5 minutes,180,176,4
1,30 minutes,180,180,0
2,2 hours,180,180,0


A short watermark delay closes windows quickly and drops more late readings; a long one keeps them all but holds every window open, in memory, for longer, and delays final results. Which readings are dropped also depends on timing: a reading that is 60 minutes late may still land in a window that is open, so the number dropped is smaller than the number of late readings.

The output mode matters too. In **update** mode (used here) Spark emits a window's running count every time it changes. In **append** mode it emits each window once, only after the watermark has passed it, so the last windows of a finite run are never emitted at all. That is easy to mistake for lost data.

A production pipeline would record dropped readings somewhere (a "late data" table) rather than lose them, and reconcile them in a later batch run.

In [12]:
spark.stop()

## Choosing between them

| | Beam | Spark |
|---|---|---|
| Mental model | A graph of transforms, run by a pluggable runner | Distributed tables with a query optimizer |
| Best for | Streaming with careful event-time semantics; Python-heavy decoding; array data via xarray-beam | Large tables: joins, aggregations, SQL; station data, catalogs, logs |
| Streaming | First-class: windows, watermarks, triggers, late data | Micro-batches with watermarks; simpler model |
| Arrays | xarray-beam | Awkward: every element must be a row |
| Where it runs | Dataflow, Flink, Spark, locally | Its own clusters (Databricks, EMR, Kubernetes), locally |

For this lab's data, the natural split is: Spark for the station tables (deduplication, completeness, joins with metadata), Beam or xarray-beam for gridded data and for a live feed of observations. Ray, covered in notebook 07, fills a third niche: running arbitrary Python functions (decoders, models) in parallel, and feeding training.

## What surprised me

**Both frameworks are lazy and both rewrite your plan**, but differently: Spark optimises a query, while Beam fuses steps and leaves the logic as written.

**Pre-aggregation is the same idea in both.** Beam's CombinePerKey and Spark's partial HashAggregate both shrink data before the shuffle, which is usually the expensive part.

**Late data is a design decision, not an edge case.** Every streaming pipeline has to choose what "final" means, and the default is to drop late data silently.

**Local runners have their own quirks.** Beam's newer default local runner (Prism) downloads a helper program; this primer selects the classic DirectRunner. Spark needs a Java runtime and timestamps with a time zone for watermarks.

## Check yourself

<details><summary>Why does Beam group by event time rather than arrival time?</summary>
Because results should describe when things happened. Grouping by arrival time would put a reading delayed by a network outage into the wrong hour, and the result would change depending on network conditions rather than on the weather.
</details>

<details><summary>In <code>explain()</code> output, what does an <code>Exchange</code> node mean, and why are there two HashAggregate steps around it?</summary>
Exchange is a shuffle: rows are moved between partitions so that equal keys end up together. Spark aggregates partially on each partition before the shuffle (so less data moves) and finishes the aggregation after it.
</details>

<details><summary>When would you use a broadcast join?</summary>
When one side of the join is small enough to copy to every worker, for example station metadata joined to billions of observations. It avoids shuffling the large side.
</details>

<details><summary>A streaming job drops 2% of readings as late. What are two ways to handle them without making every result wait?</summary>
Keep the low-latency streaming result, and (1) route late readings to a side table and fold them in with a periodic batch recomputation, or (2) allow lateness with a trigger that emits corrected (late) panes, accepting more state and updated results downstream.
</details>

> **In a design discussion.** Pick the framework per source and say why: Spark for tabular observations and metadata, Beam (or xarray-beam) for gridded data and for streams with late data, and Ray for Python-heavy decoding and feeding training. Then talk about the shuffle (what moves, how to shrink it), idempotent writes, and how late or corrected data is handled.